In [ ]:
import pandas as pd
import re
from sqlalchemy import create_engine
import urllib
import os
from dotenv import load_dotenv

load_dotenv()

In [ ]:
params = urllib.parse.quote_plus(
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={os.getenv('DB_SERVER')};"
    f"DATABASE={os.getenv('DB_NAME')};"
    f"UID={os.getenv('DB_USER')};"
    f"PWD={os.getenv('DB_PASSWORD')};"
)

# Crear el motor de conexión
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

In [3]:
query = """
SELECT DISTINCT c.CLICodigo, 
    c.CLINombres, 
    c.CLIApellidos, 
    c.CLIEmailPrincipal, 
    c.CLICelular,
    c.CLITipoDocumento
FROM Ventas_Comerssia.dbo.Clientes_Comerssia c

"""

# Ejecutar y cargar en DataFrame
df = pd.read_sql(query, engine)


# # Obtener los CLICodigos ya existentes en la tabla SQL
# clientes_existentes = pd.read_sql("SELECT CLICodigo FROM Contacto_Clientes", con=engine)

In [4]:
#Limpiar Celular
def limpiar_celular(cel):
    if not cel:
        return ""
    cel = str(cel).strip().replace(" ", "").replace("-", "").replace("(", "").replace(")", "")
    if cel.startswith("+57"):
        cel = cel[3:]
    elif cel.startswith("57") and len(cel) > 10:
        cel = cel[2:]
    return cel

df['CLICelular'] = df['CLICelular'].apply(limpiar_celular)

#Validar celular
def es_celular_valido(cel):
    return bool(re.fullmatch(r"3\d{9}", str(cel)))

df['CelularValido'] = df['CLICelular'].apply(es_celular_valido)

#Limpiar y validar email
df['CLIEmailPrincipal'] = df['CLIEmailPrincipal'].apply(lambda x: str(x).strip().lower())

def es_email_valido(email):
    email = str(email).strip().lower()

    # Si está vacío
    if not email:
        return False
    
    # Palabras prohibidas
    palabras_no_permitidas = ["negad", "pendi"]
    if any(palabra in email for palabra in palabras_no_permitidas):
        return False
    
    # Validación de formato
    return bool(re.fullmatch(r'^[\w\.-]+@[\w\.-]+\.\w+$', email))

df['EmailValido'] = df['CLIEmailPrincipal'].apply(es_email_valido)

In [5]:
def limpiar_codigo(codigo):
    codigo = str(codigo).strip()     
    codigo = re.sub(r'^C\s*', '', codigo, flags=re.IGNORECASE)         # quita espacios al inicio y al final
    codigo = codigo.replace(' ', '')          # elimina cualquier espacio restante en el medio
    return codigo

df.insert(1, 'Documento', df['CLICodigo'].apply(limpiar_codigo))

df

,CLICodigo,Documento,CLINombres,CLIApellidos,CLIEmailPrincipal,CLICelular,CLITipoDocumento,CelularValido,EmailValido
0,None,None,HELENA,CASTAÑO,helenacastanog@gmail.com,3146175933,CC,True,True
1,1000788466,1000788466,None,,none,,CC,False,False
2,1002542697,1002542697,MARIA JULIANA,OROZCO GONZALEZ,julianaog13@gmail.com,3127160903,CC,True,True
3,1015429541,1015429541,DANIELA,ORJUELA,none,,CC,False,False
4,1018500677,1018500677,None,,none,,CC,False,False
...,...,...,...,...,...,...,...,...,...
382774,CY04HC6XMC,Y04HC6XMC,AMARANTHA,MESU JOYA,roemer.amarantha@gmail.com,1111111111,CC,False,True
382775,CYAMILE USME,YAMILEUSME,CÉSAR,ARÉVALO OCHOA,sinergia75@hotmail.com,3143336439,CC,True,True
382776,Cyc004603,yc004603,SILVA,BRUNA,pendiente@provenzal.net,1111111111,CC,False,False
382777,CYI024605,YI024605,GERUSA,TEIXEIRA,negado@provenzal.net,1111111111,CC,False,False


In [6]:
# Detectar duplicados
cel_duplicados = df['CLICelular'].duplicated(keep=False)
email_duplicados = df['CLIEmailPrincipal'].duplicated(keep=False)

# Marcar celulares duplicados como no válidos
df.loc[cel_duplicados, 'CelularValido'] = False

# Marcar correos duplicados como no válidos
df.loc[email_duplicados, 'EmailValido'] = False

In [7]:
#Agregar columna contacto
def determinar_contacto(row):
    if row['CelularValido'] and row['EmailValido']:
        return "Cel + Email"
    elif row['CelularValido']:
        return "Cel"
    elif row['EmailValido']:
        return "Email"
    else:
        return "Falso"

df['Contacto'] = df.apply(determinar_contacto, axis=1)

In [8]:
# Filtrar para dejar solo nuevos clientes
# df_nuevos = df[~df["CLICodigo"].isin(clientes_existentes["CLICodigo"])]

In [9]:
# # Insertar solo los nuevos en SQL
# if not df.empty:
#     df_nuevos.to_sql("Contacto_Clientes", con=engine, if_exists="append", index=False)
#     print(f"{len(df_nuevos)} nuevos clientes insertados.")
# else:
#     print("No hay nuevos clientes por insertar.")

In [10]:
# # Exportar todos los registros con validaciones incluidas
# df.to_excel("clientes_con_validacion.xlsx", index=False)

In [11]:
df.to_sql("Contacto_Clientes", engine, if_exists="replace", index=False)

100